In [1]:
import pandas as pd
import json

def process_rich_aggregated_routes():
    # 1. 데이터 로드
    try:
        loc_df = pd.read_csv('location_list2.csv')
        ygs_df = pd.read_csv('dataset_merged_ok_all.csv')
    except FileNotFoundError:
        print("CSV 파일을 찾을 수 없습니다.")
        return

    # 2. 데이터 타입 통일 (Merge 오류 방지)
    loc_df['location_id'] = loc_df['location_id'].astype(str)
    ygs_df['location_id'] = ygs_df['location_id'].astype(str)
    
    # 3. 데이터 병합 (위경도 + 장소명 가져오기)
    # ygs에는 장소명이 없을 수도 있으니 location_list2의 scan_location을 메인으로 사용
    merged_df = pd.merge(ygs_df, 
                         loc_df[['location_id', 'lat', 'lon', 'scan_location']], 
                         on='location_id', 
                         how='left')
    
    # 병합 후 scan_location 컬럼이 중복될 경우 정리 (loc_df의 것을 우선 사용)
    if 'scan_location_y' in merged_df.columns:
        merged_df['scan_location'] = merged_df['scan_location_y']
    elif 'scan_location_x' in merged_df.columns:
        merged_df['scan_location'] = merged_df['scan_location_x']

    # 좌표 없는 데이터 제거
    merged_df = merged_df.dropna(subset=['lat', 'lon'])

    # 4. 시간 정렬 (경로 추적의 핵심)
    merged_df['event_time'] = pd.to_datetime(merged_df['event_time'])
    merged_df = merged_df.sort_values(by=['epc_code', 'event_time'])

    # 5. 경로 집계 (Aggregation)
    # Key: (출발지ID, 도착지ID) 튜플
    # Value: 횟수, 좌표정보, 장소정보, EPC리스트
    routes_dict = {}

    for epc, group in merged_df.groupby('epc_code'):
        if len(group) < 2: continue
        
        # DataFrame을 순회하기 쉽게 리스트로 변환 (속도 최적화)
        records = group.to_dict('records')
        
        for i in range(len(records) - 1):
            start = records[i]
            end = records[i+1]
            
            # 제자리 이동(A -> A)은 지도상 표현 불가하므로 스킵 (필요시 주석 처리)
            if start['location_id'] == end['location_id']:
                continue

            # 고유 키 생성 (ID 기준)
            route_key = (start['location_id'], end['location_id'])

            if route_key not in routes_dict:
                # 새로운 경로 발견 시 초기화
                routes_dict[route_key] = {
                    "count": 0,
                    "source_info": {
                        "id": start['location_id'],
                        "name": start.get('scan_location', 'Unknown'),
                        "coords": [start['lon'], start['lat']] # Deck.gl 순서 [lon, lat]
                    },
                    "target_info": {
                        "id": end['location_id'],
                        "name": end.get('scan_location', 'Unknown'),
                        "coords": [end['lon'], end['lat']]
                    },
                    "epc_list": set() # 중복 방지를 위해 set 사용
                }
            
            # 데이터 업데이트
            routes_dict[route_key]["count"] += 1
            routes_dict[route_key]["epc_list"].add(epc)

    # 6. JSON 구조로 변환
    output_data = []
    for key, val in routes_dict.items():
        val['epc_list'] = list(val['epc_list']) # set을 list로 변환 (JSON 직렬화 위해)
        output_data.append(val)

    # 7. 파일 저장
    output_filename = 'rich_routes.json'
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    print(f"처리 완료! '{output_filename}' 생성됨.")
    print(f"총 추출된 고유 경로(Segment) 수: {len(output_data)}")

if __name__ == "__main__":
    process_rich_aggregated_routes()


처리 완료! 'rich_routes.json' 생성됨.
총 추출된 고유 경로(Segment) 수: 95
